### Renombrar columnas a snake_case en bronze

Observaciones:
- Al cargar bronze, las columnas se renombran a snake_case sin tildes ni caracteres especiales (`Ajuste de precio ($)` → `ajuste_precio`). Los valores no se modifican
- Delta no acepta espacios ni caracteres especiales en nombres de columna. Se descartó column mapping porque obliga a usar backticks en todas las consultas y arrastra errores del archivo (espacio al final de `Precio Original `)
- Bronze no es 100 % fiel al archivo en los nombres; la fidelidad completa la garantiza el Excel original en landing. Si el negocio cambia un encabezado, la carga falla (deseable). Si agrega una columna nueva, se ignora sin aviso: hay que revisar el encabezado del archivo en cada carga nueva

In [0]:
%sql
CREATE OR REPLACE TABLE fabriperezchauca.bronze.ventas
COMMENT 'Ventas tal como vienen en el CSV. Todas las columnas string. Grano: una fila = una fila del archivo'
AS
SELECT
    Fecha                     AS fecha,
    Cantidad                  AS cantidad,
    Producto                  AS producto,
    Talla                     AS talla,
    ` Precio Dado `             AS precio_dado,
    ` Ajuste de precio ($) `  AS ajuste_precio,
    ` Precio Original  `      AS precio_original,
    ` Precio Total `            AS precio_total,
    `Forma de Pago`           AS forma_pago,
    Regateo                   AS regateo,
    `Categoría`               AS categoria,
    Observaciones             AS observaciones,
    _rescued_data,
    _metadata.file_path              AS _archivo_origen,
    _metadata.file_modification_time AS _fecha_archivo,
    current_timestamp()              AS _fecha_ingesta
FROM read_files(
    '/Volumes/fabriperezchauca/landing/archivos_crudos/venta-fabriperez-chauca-2026-06.csv',
    format => 'csv',
    header => true,
    inferColumnTypes => false
);

In [0]:
%sql
select * from fabriperezchauca.bronze.ventas limit 10

In [0]:
%sql
describe table fabriperezchauca.bronze.ventas